In [3]:
import pandas as pd


In [4]:
df = pd.read_csv(r"D:\projects\automobile rag\data\OBD-II Driving Data - Classified.csv", encoding='latin-1')


In [5]:
# Rename by position - no matching issues
df.columns = [
    'battery_voltage', 'engine_load_pct', 'coolant_temp_c',
    'fuel_trim_short_pct', 'fuel_trim_long_pct', 'intake_pressure_kpa',
    'engine_rpm', 'speed_kmh', 'ignition_timing_deg', 'intake_air_temp_c',
    'throttle_position_pct', 'lambda_sensor_voltage', 'distance_mil_on_km',
    'evap_purge_pct', 'fuel_level_pct', 'warmups_since_clear',
    'distance_since_clear_km', 'barometric_pressure_kpa', 'o2_equivalence_ratio',
    'o2_sensor_current_ma', 'catalyst_temp_c', 'control_module_voltage',
    'absolute_load_pct', 'commanded_afr', 'relative_throttle_pct',
    'ambient_temp_c', 'throttle_position_b_pct', 'accelerator_pedal_d_pct',
    'accelerator_pedal_e_pct', 'commanded_throttle_actuator_pct',
    'fuel_trim_long_bank2_pct', 'steering_angle_deg', 'steering_angle_speed',
    'label', 'driver_id'
]

print(df.columns.tolist())
print(df.shape)

['battery_voltage', 'engine_load_pct', 'coolant_temp_c', 'fuel_trim_short_pct', 'fuel_trim_long_pct', 'intake_pressure_kpa', 'engine_rpm', 'speed_kmh', 'ignition_timing_deg', 'intake_air_temp_c', 'throttle_position_pct', 'lambda_sensor_voltage', 'distance_mil_on_km', 'evap_purge_pct', 'fuel_level_pct', 'warmups_since_clear', 'distance_since_clear_km', 'barometric_pressure_kpa', 'o2_equivalence_ratio', 'o2_sensor_current_ma', 'catalyst_temp_c', 'control_module_voltage', 'absolute_load_pct', 'commanded_afr', 'relative_throttle_pct', 'ambient_temp_c', 'throttle_position_b_pct', 'accelerator_pedal_d_pct', 'accelerator_pedal_e_pct', 'commanded_throttle_actuator_pct', 'fuel_trim_long_bank2_pct', 'steering_angle_deg', 'steering_angle_speed', 'label', 'driver_id']
(555000, 35)


In [6]:
df.to_csv(r"D:\projects\automobile rag\data\obd_clean.csv", index=False)
print("✅ Saved obd_clean.csv")

✅ Saved obd_clean.csv


In [7]:
df_check = pd.read_csv(r"D:\projects\automobile rag\data\obd_clean.csv")
print(df_check.shape)
print(df_check.columns.tolist())
print(df_check.head(3))

(555000, 35)
['battery_voltage', 'engine_load_pct', 'coolant_temp_c', 'fuel_trim_short_pct', 'fuel_trim_long_pct', 'intake_pressure_kpa', 'engine_rpm', 'speed_kmh', 'ignition_timing_deg', 'intake_air_temp_c', 'throttle_position_pct', 'lambda_sensor_voltage', 'distance_mil_on_km', 'evap_purge_pct', 'fuel_level_pct', 'warmups_since_clear', 'distance_since_clear_km', 'barometric_pressure_kpa', 'o2_equivalence_ratio', 'o2_sensor_current_ma', 'catalyst_temp_c', 'control_module_voltage', 'absolute_load_pct', 'commanded_afr', 'relative_throttle_pct', 'ambient_temp_c', 'throttle_position_b_pct', 'accelerator_pedal_d_pct', 'accelerator_pedal_e_pct', 'commanded_throttle_actuator_pct', 'fuel_trim_long_bank2_pct', 'steering_angle_deg', 'steering_angle_speed', 'label', 'driver_id']
   battery_voltage  engine_load_pct  coolant_temp_c  fuel_trim_short_pct  \
0             13.1             18.0              87                  0.0   
1             13.1             17.6              87                 

In [8]:
# Split into sessions of 1000 rows each
df_check['session_id'] = (df_check.index // 1000).astype(str)
df_check['session_id'] = 'S' + df_check['session_id'].str.zfill(4)

print(df_check['session_id'].value_counts().head(5))
print(f"Total sessions: {df_check['session_id'].nunique()}")

session_id
S0554    1000
S0000    1000
S0001    1000
S0002    1000
S0003    1000
Name: count, dtype: int64
Total sessions: 555


In [9]:
sessions = df_check.groupby(['session_id', 'driver_id']).agg(
    avg_rpm=('engine_rpm', 'mean'),
    max_rpm=('engine_rpm', 'max'),
    avg_speed=('speed_kmh', 'mean'),
    max_speed=('speed_kmh', 'max'),
    avg_engine_load=('engine_load_pct', 'mean'),
    max_coolant_temp=('coolant_temp_c', 'max'),
    avg_throttle=('throttle_position_pct', 'mean'),
    max_steering_angle=('steering_angle_deg', 'max'),
    avg_fuel_level=('fuel_level_pct', 'mean'),
    avg_battery_voltage=('battery_voltage', 'mean'),
    aggressive_event_count=('label', 'sum'),
    total_events=('label', 'count'),
).reset_index()

# Driving label — if more than 50% events are aggressive, session is aggressive
sessions['driving_label'] = sessions.apply(
    lambda r: 'aggressive' if r['aggressive_event_count'] / r['total_events'] > 0.5 else 'moderate',
    axis=1
)

print(sessions.shape)
print(sessions.head(3))

(557, 15)
  session_id  driver_id   avg_rpm  max_rpm  avg_speed  max_speed  \
0      S0000          1  1659.257     3048     21.874         46   
1      S0001          1  2067.106     3023     26.684         46   
2      S0002          1  2187.113     3078     30.402         63   

   avg_engine_load  max_coolant_temp  avg_throttle  max_steering_angle  \
0          30.6135                93       18.6346              135.86   
1          32.1858                90       23.9049              233.73   
2          32.9020                90       26.1289              137.70   

   avg_fuel_level  avg_battery_voltage  aggressive_event_count  total_events  \
0            73.7              13.1000                     978          1000   
1            73.7              13.0968                     922          1000   
2            73.7              13.1000                     871          1000   

  driving_label  
0    aggressive  
1    aggressive  
2    aggressive  


In [10]:
sessions.to_csv(r"D:\projects\automobile rag\data\sessions_clean.csv", index=False)
print("✅ Saved sessions_clean.csv")
print(f"Aggressive sessions: {(sessions['driving_label'] == 'aggressive').sum()}")
print(f"Moderate sessions: {(sessions['driving_label'] == 'moderate').sum()}")

✅ Saved sessions_clean.csv
Aggressive sessions: 544
Moderate sessions: 13
